# Tome 3, Chapter 9: Poisson Processes (Python companion)

Exercises first, solutions at the end. We build the Poisson process in three ways and check that they agree, simulate traffic with a rate that varies over the day by thinning, measure the inspection paradox at a bus stop, compute exact distances to the Poisson law to test Le Cam's inequality, simulate an M/M/1 queue to watch the mean queue length explode near full load, and see the Gumbel law appear in the coupon collector.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(9)

## Exercises

**P9.1 (★) Three constructions.** Build a Poisson process of rate $\lambda=3$ on $[0,10]$ in three ways: (a) from exponential gaps (Definition 9.3); (b) by drawing $N\sim$ Poisson$(30)$ and then $N$ sorted uniform points (Theorem 9.4); (c) by cutting $[0,10]$ into $10^4$ slots, each holding a point with probability $\lambda h$ (definition (C) of Theorem 9.7). Compare the laws of $N(10)$, of $N(2)$, and of the first gap.

**P9.2 (★★) Traffic over a day by thinning.** Requests to a website arrive with rate $\lambda(t)=100\bigl(1-\cos\frac{2\pi t}{24}\bigr)$ per hour ($t$ in hours, quiet at midnight, busy at noon). Simulate one day by thinning a process of rate $200$ (keep a point at time $t$ with probability $\frac{\lambda(t)}{200}$), check that the expected number of requests is $\int_0^{24}\lambda=2400$, and plot the histogram of arrival hours against $\lambda$.

**P9.3 (★) The bus stop.** Buses arrive as a Poisson process with mean gap $10$ minutes. Passengers arrive at uniform random times over a long day. Measure the mean gap that a passenger falls into and the mean waiting time (Proposition 9.10). Repeat with gaps uniform on $[0,20]$ and with a strict timetable (gaps $10$), and compare with Exercise 9.6.

**P9.4 (★★) Le Cam's inequality, exactly.** For independent Bernoulli$(p_i)$, compute the exact law of $S=\sum X_i$ by dynamic programming and its total variation distance to Poisson$(\mu)$. Compare with Le Cam's bound $\sum p_i^2$ (Theorem 9.11) and with the Chen–Stein bound $\frac{1-e^{-\mu}}\mu\sum p_i^2$, for binomial$(n,\frac5n)$ with $n=10,\dots,10^4$, and for the advertisement example ($10^6$ users).

**P9.5 (★★) The M/M/1 queue.** Write an event-driven simulation of the M/M/1 queue with $\mu=1$. For $\rho=0.5,0.8,0.9,0.95$, measure the time-average queue length and the mean time spent in the system by customers, and compare with $\frac\rho{1-\rho}$ and $\frac1{\mu-\lambda}$ (Theorem 9.12, Exercise 9.5). How long must the simulation run for the estimate to be reliable?

**P9.6 (★★) The Gumbel law.** For $n=500$ coupon types arriving as a Poisson process of rate $1$ (Exercise 9.4), simulate $T$ many times, plot the histogram of $\frac{T-n\ln n}n$ against the Gumbel density $e^{-c}e^{-e^{-c}}$, and compare $\mathbb E[T]$ with $nH_n$. Do the same for the discrete collector's time $K$.

## Solutions

**P9.1.**

In [ ]:
lam, T, reps = 3.0, 10.0, 20000
def by_gaps():
    t, pts = 0.0, []
    while True:
        t += rng.exponential(1 / lam)
        if t > T: return np.array(pts)
        pts.append(t)
def by_uniform():
    return np.sort(rng.uniform(0, T, rng.poisson(lam * T)))
h = T / 10_000
def by_slots():
    return (np.flatnonzero(rng.random(10_000) < lam * h) + 1) * h
for name, f in [('gaps', by_gaps), ('uniform', by_uniform), ('slots', by_slots)]:
    S = [f() for _ in range(reps)]
    n10 = np.array([len(s) for s in S]); n2 = np.array([np.sum(s <= 2) for s in S])
    g = np.array([s[0] for s in S if len(s)])
    print(f'{name:8s} E N(10)={n10.mean():6.3f} Var={n10.var():6.3f}   E N(2)={n2.mean():.3f} Var={n2.var():.3f}   first gap mean={g.mean():.4f}')
print('theory   E N(10)=30, Var=30;  E N(2)=6, Var=6;  first gap mean=1/3 (slightly less when conditioned on a point before 10)')

The three constructions agree to within sampling error. The slot construction is a Bernoulli process, so its counts are binomial: the variance of $N(10)$ is $30(1-\lambda h)=29.99$, which simulation cannot tell from $30$. This is the content of Theorem 9.7: the three descriptions are of the same object, and in practice one picks the one that is cheapest. Construction (b) is the fastest in `numpy` because it needs no loop.

**P9.2.**

In [ ]:
lmax = 200.0
lam_t = lambda t: 100 * (1 - np.cos(2 * np.pi * t / 24))
def one_day():
    n = rng.poisson(lmax * 24)
    t = rng.uniform(0, 24, n)
    return np.sort(t[rng.random(n) < lam_t(t) / lmax])
days = [one_day() for _ in range(2000)]
counts = np.array([len(d) for d in days])
print('mean requests per day:', counts.mean(), '  variance:', counts.var(), '  theory: 2400 and 2400')
allt = np.concatenate(days[:200])
plt.figure(figsize=(7, 3))
plt.hist(allt, bins=48, density=True, alpha=0.6, label='simulated arrival hours')
tt = np.linspace(0, 24, 300)
plt.plot(tt, lam_t(tt) / 2400, 'k', label='$\\lambda(t)/2400$')
plt.xlabel('hour'); plt.legend(); plt.tight_layout(); plt.show()

The count over a day is Poisson$(2400)$ (mean and variance agree), and the arrival hours follow the shape of $\lambda$. The thinning was done in one vectorised pass: draw the homogeneous process of rate $\lambda_{\max}$ by construction (b), then keep each point with probability $\frac{\lambda(t)}{\lambda_{\max}}$. This is the standard way load generators and network simulators produce realistic traffic; the waste is the fraction of rejected points, here $\frac12$.

**P9.3.** A passenger arriving at a uniform time falls in a gap with probability proportional to its length, and then waits a uniform fraction of it.

In [ ]:
def observe(gaps, npass=200_000):
    ends = np.cumsum(gaps)
    t = rng.uniform(0, ends[-1] * 0.99, npass)
    i = np.searchsorted(ends, t)
    starts = ends - gaps
    return gaps[i].mean(), (ends[i] - t).mean()
n = 200_000
for name, gaps in [('Poisson', rng.exponential(10, n)), ('uniform[0,20]', rng.uniform(0, 20, n)), ('timetable', np.full(n, 10.0))]:
    L, W = observe(gaps)
    print(f'{name:14s} mean gap {gaps.mean():6.2f}   gap seen by a passenger {L:6.2f}   mean wait {W:6.2f}')
print('theory: Poisson 20 and 10; uniform 40/3 = 13.33 and 6.67; timetable 10 and 5')

With Poisson buses, a passenger waits $10$ minutes on average, as long as the mean gap, although the buses come every $10$ minutes on average: the gap they land in has mean $20$. With uniform gaps the wait is $\frac{20}3\approx6.7$, with a timetable $5$. Regularity halves waiting times, which is why transport operators fight bunching. The same effect biases any measurement made at a random time: the job you find running on a server is longer than the typical job, and the file you find being downloaded is larger than the typical file.

**P9.4.** The law of $S$ is built one variable at a time: $P(S_{i}=k)=(1-p_i)P(S_{i-1}=k)+p_iP(S_{i-1}=k-1)$. For $10^6$ variables we truncate the support where the mass is negligible.

In [ ]:
def poisson_pmf(mu, K):
    k = np.arange(K)
    return np.exp(-mu + k * np.log(mu) - np.array([math.lgamma(j + 1) for j in k]))
def tv_to_poisson(p, K=None):
    mu = p.sum()
    K = K or int(mu + 12 * math.sqrt(mu) + 30)
    f = np.zeros(K); f[0] = 1.0
    for q in p:
        f[1:] = (1 - q) * f[1:] + q * f[:-1]
        f[0] *= (1 - q)
    g = poisson_pmf(mu, K)
    return 0.5 * (np.abs(f - g).sum() + (1 - g.sum()) + (1 - f.sum()))
print('   n    exact TV    Le Cam     Chen-Stein')
for n in [10, 100, 1000, 10000]:
    p = np.full(n, 5 / n); s = (p**2).sum()
    print(f'{n:5d}   {tv_to_poisson(p):.6f}   {s:.6f}   {(1 - math.exp(-5)) / 5 * s:.6f}')

In [ ]:
p = rng.uniform(1e-6, 1e-4, 1_000_000)
p *= 50 / p.sum()
print('max p:', p.max(), ' sum p:', p.sum(), ' Le Cam bound:', (p**2).sum())
print('Chen-Stein bound:', (1 - math.exp(-50)) / 50 * (p**2).sum())
print('exact TV distance:', tv_to_poisson(p))

For the binomial, the exact distance is about $\frac{1.2}n$ here, Le Cam's bound is $\frac{25}n$, and the Chen–Stein bound about $\frac5n$: all three decay like $\frac1n$, and Le Cam is off by a constant factor that grows with $\mu$. For the advertisement example the probabilities lie between $10^{-6}$ and about $10^{-4}$; Le Cam's bound is about $3\times10^{-3}$, under the $5\times10^{-3}$ of the text, the Chen–Stein bound is about $7\times10^{-5}$, and the true distance is about $1.6\times10^{-5}$. The dynamic programme costs $O(nK)$ operations, which is why the Poisson approximation, which costs nothing, is used in practice: one of the bounds then certifies it.

**P9.5.** Event-driven simulation: from state $q$, the next event comes after an Exp$(\lambda+\mu\mathbf 1_{q>0})$ time and is an arrival with probability $\frac\lambda{\lambda+\mu\mathbf 1_{q>0}}$. We record the area under $Q$, and the arrival and departure times of customers (first in, first out).

In [ ]:
def mm1(lam, mu, horizon):
    t, q, area = 0.0, 0, 0.0
    arrivals, sojourn = [], []
    head = 0
    while t < horizon:
        rate = lam + (mu if q else 0.0)
        dt = rng.exponential(1 / rate)
        area += q * dt; t += dt
        if rng.random() < lam / rate:
            q += 1; arrivals.append(t)
        else:
            q -= 1; sojourn.append(t - arrivals[head]); head += 1
    return area / t, np.mean(sojourn), len(arrivals) / t
print(' rho   mean Q   theory     mean time   theory    lambda*W')
for rho in [0.5, 0.8, 0.9, 0.95]:
    L, W, lam_hat = mm1(rho, 1.0, 200_000)
    print(f'{rho:4.2f}   {L:6.2f}   {rho / (1 - rho):6.2f}     {W:6.2f}     {1 / (1 - rho):6.2f}    {lam_hat * W:6.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for rho, c in [(0.8, 'C0'), (0.95, 'C1')]:
    for _ in range(3):
        t, q, area, xs, ys = 0.0, 0, 0.0, [], []
        while t < 20000:
            rate = rho + (1.0 if q else 0.0)
            dt = rng.exponential(1 / rate); area += q * dt; t += dt
            q += 1 if rng.random() < rho / rate else -1
            if len(xs) == 0 or t - xs[-1] > 50:
                xs.append(t); ys.append(area / t)
        ax.plot(xs, ys, c, lw=0.8)
    ax.axhline(rho / (1 - rho), color=c, ls='--')
ax.set_xlabel('simulated time'); ax.set_ylabel('running mean of Q'); ax.set_ylim(0, 40)
plt.tight_layout(); plt.show()

At low load the measured means match $\frac\rho{1-\rho}$ and $\frac1{1-\rho}$, and $\lambda W$ matches $L$ in every row (Little's law holds path by path, up to the customers still in the system at the end). But at $\rho=0.9$ and $0.95$ the estimates are off by $5$ to $15\%$ even after $200\,000$ time units, and rerunning with another seed moves them by as much. The running means show why: near saturation the queue makes long excursions (Figure 9.3), and the relaxation time of the M/M/1 queue grows like $(1-\rho)^{-2}$. A load test that runs for one minute on a system at $95\%$ load measures nothing. This is the practical lesson of the chapter's opening question: at $90\%$ load about $9$ requests are in the system, at $99\%$ about $99$, and in both cases the answer takes a long time to show up in measurements.

**P9.6.**

In [ ]:
n, reps = 500, 4000
H = sum(1 / k for k in range(1, n + 1))
Ts, Ks = np.empty(reps), np.empty(reps)
for r in range(reps):
    F = rng.exponential(n, n)                # first arrival time of each type, Exp(1/n)
    Ts[r] = F.max()
    # discrete time: coupon k+1 after a geometric number of draws with success (n-k)/n
    Ks[r] = rng.geometric((n - np.arange(n)) / n).sum()
print('E[T] simulated', Ts.mean(), '  n H_n =', n * H)
print('E[K] simulated', Ks.mean(), '  n H_n =', n * H)
c = np.linspace(-3, 7, 300)
plt.figure(figsize=(7, 3))
plt.hist((Ts - n * math.log(n)) / n, bins=60, density=True, alpha=0.5, label='continuous T')
plt.hist((Ks - n * math.log(n)) / n, bins=60, density=True, alpha=0.5, label='discrete K')
plt.plot(c, np.exp(-c - np.exp(-c)), 'k', label='Gumbel')
plt.xlabel('(time - n ln n)/n'); plt.legend(); plt.tight_layout(); plt.show()

Both histograms follow the Gumbel density, and both means are close to $nH_n=n\ln n+\gamma n+O(1)$, where $\gamma\approx0.577$ is the mean of the Gumbel law. In continuous time, $T$ is a maximum of $n$ independent exponentials, one line of `numpy`; the discrete collector needs the sum of $n$ geometric variables. Poissonisation turned a dependent problem into an independent one, and costs nothing in the limit: $T=E_1+\dots+E_K$ and $\frac{T-K}n\to0$. This is the standard trick behind the analysis of hashing with chaining and of balls in bins: throw a Poisson number of balls, and the bins become independent.